# Debug Craftax Model

Load model checkpoints, run on each world seed, and compare success rates.
Use this to debug why models that achieve 100% success on H100 may fail on Mac.

In [1]:
import os
import sys
import pickle
import functools
from glob import glob

# Ensure project root is on sys.path and is cwd
if os.path.basename(os.getcwd()) == "data_processing":
  PROJECT_ROOT = os.path.dirname(os.getcwd())
else:
  PROJECT_ROOT = os.getcwd()
os.chdir(PROJECT_ROOT)
sys.path.insert(0, PROJECT_ROOT)
sys.path.insert(0, os.path.join(PROJECT_ROOT, "simulations"))

import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from jaxneurorl.agents import value_based_basics as vbb
from jaxneurorl.agents import qlearning

from data_processing.process_model_data import (
  load_craftax_environment,
  load_algorithm_ckpt_params,
  make_epsilon_greedy_actor,
)
from data_processing.utils_craftax import get_agent_position
from data_processing.utils import get_in_episode, EpisodeData
from simulations import craftax_simulation_configs
from simulations.craftax_utils import (
  render_fn,
  place_arrows_on_image,
  actions_from_path,
)
import data_configs

Loading textures from cache.
Textures successfully loaded from cache.
Cache dir: /Users/wilka/git/research/multitask_preplay/experiments/craftax/craftax_cache
regular map cache already exists in craftax library
full map cache already exists in craftax library
Loading textures from cache.
Textures successfully loaded from cache.


In [2]:
# ---- Configuration ----
MODEL_NAME = "preplay"  # one of: "qlearning", "usfa", "dyna", "preplay"
DATA_DIR = data_configs.CRAFTAX_DATA_DIR
EPSILON = 0.0  # 0.0 = pure greedy (matches server greedy); server eval uses 0.1; notebook previously used 0.2
NUM_EPISODES = 10  # episodes per world seed per ckpt (server uses ~20+ via autoreset)
NUM_RNG_SEEDS = (
  5  # number of RNG seeds to average over (reduces dependence on a single random seed)
)

In [3]:
# ---- JAX Diagnostics ----
print(f"JAX version: {jax.__version__}")
print(f"Backend: {jax.default_backend()}")
print(f"Devices: {jax.devices()}")
print(f"Default float dtype: {jnp.zeros(1).dtype}")
print(f"Default int dtype: {jnp.zeros(1, dtype=int).dtype}")
print(f"x64 enabled: {jax.config.x64_enabled}")

JAX version: 0.4.22
Backend: cpu
Devices: [CpuDevice(id=0)]
Default float dtype: float32
Default int dtype: int32
x64 enabled: False


In [4]:
# ---- Load Environment ----
landmark_features = MODEL_NAME == "usfa"
env, example_timestep, example_env_params = load_craftax_environment(
  landmark_features=landmark_features
)
print(f"Environment loaded (landmark_features={landmark_features})")

Environment loaded (landmark_features=False)


In [5]:
# ---- Discover all checkpoint seeds ----
ckpt_dirs = sorted(glob(os.path.join(DATA_DIR, MODEL_NAME, "seed=*")))
ckpt_seeds = []
for d in ckpt_dirs:
  seed = int(os.path.basename(d).split("=")[1])
  safetensors = os.path.join(d, f"{MODEL_NAME}.safetensors")
  if os.path.exists(safetensors):
    ckpt_seeds.append(seed)
  else:
    print(f"  Skipping seed={seed} (no {MODEL_NAME}.safetensors)")
ckpt_seeds = sorted(ckpt_seeds)
print(f"Found {len(ckpt_seeds)} valid checkpoints: seeds={ckpt_seeds}")

train_configs = craftax_simulation_configs.TRAIN_EVAL_CONFIGS
test_configs = craftax_simulation_configs.TEST_CONFIGS
rng = jax.random.PRNGKey(0)

  Skipping seed=14 (no preplay.safetensors)
  Skipping seed=15 (no preplay.safetensors)
  Skipping seed=16 (no preplay.safetensors)
Found 10 valid checkpoints: seeds=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10]


In [6]:
# ---- Set up algorithm once, reuse JIT across seeds ----
MODEL_TO_IMPORTS = {
  "qlearning": (
    "simulations.qlearning_craftax",
    "make_multigoal_craftax_agent",
    "make_optimizer",
  ),
  "usfa": (
    "simulations.usfa_craftax",
    "make_multigoal_craftax_agent",
    "make_optimizer",
  ),
  "dyna": ("simulations.dyna_craftax", "make_agent", "make_optimizer"),
  "preplay": (
    "simulations.multitask_preplay_craftax_v2",
    "make_craftax_multigoal_agent",
    "make_optimizer",
  ),
}

import importlib

mod_name, agent_fn_name, opt_fn_name = MODEL_TO_IMPORTS[MODEL_NAME]
mod = importlib.import_module(mod_name)
_make_agent_fn = getattr(mod, agent_fn_name)
_make_optimizer = getattr(mod, opt_fn_name)

if MODEL_NAME == "usfa":
  from simulations.craftax_web_env import active_task_vectors

  all_tasks = jnp.concatenate(
    (active_task_vectors, jnp.zeros_like(active_task_vectors)), axis=-1
  )
  _make_agent_fn = functools.partial(_make_agent_fn, all_tasks=all_tasks)

# Load first seed's config to set up architecture
first_ckpt_path = os.path.join(DATA_DIR, MODEL_NAME, f"seed={ckpt_seeds[0]}")
first_params = load_algorithm_ckpt_params(
  os.path.join(first_ckpt_path, f"{MODEL_NAME}.safetensors")
)
with open(os.path.join(first_ckpt_path, f"{MODEL_NAME}.config"), "rb") as f:
  arch_config = pickle.load(f)

arch_config["NUM_ENVS"] = NUM_EPISODES
MAX_STEPS = 300


def vmap_reset(rng, env_params):
  return jax.vmap(env.reset, in_axes=(0, None))(
    jax.random.split(rng, NUM_EPISODES), env_params
  )


def vmap_step(rng, env_state, action, env_params):
  return jax.vmap(env.step, in_axes=(0, 0, 0, None))(
    jax.random.split(rng, NUM_EPISODES), env_state, action, env_params
  )


setup_rng = jax.random.PRNGKey(arch_config["SEED"])
setup_rng, setup_rng_ = jax.random.split(setup_rng)
_example_ts = vmap_reset(setup_rng_, example_env_params)

agent, _init_params, reset_fn = _make_agent_fn(
  config=arch_config,
  env=env,
  env_params=example_env_params,
  example_timestep=_example_ts,
  rng=setup_rng_,
)
setup_rng, setup_rng_ = jax.random.split(setup_rng)
actor = make_epsilon_greedy_actor(
  config=arch_config, agent=agent, rng=setup_rng_, epsilon=EPSILON
)

# Create initial train_state (we'll .replace() params for each seed)
base_train_state = vbb.CustomTrainState.create(
  apply_fn=agent.apply,
  params=first_params,
  target_network_params=first_params,
  tx=_make_optimizer(arch_config),
)


# JIT-compiled eval that takes train_state as argument (compiled once, reused)
@jax.jit
def eval_episode(rng, env_params, train_state):
  rng, rng_ = jax.random.split(rng)
  init_timestep = vmap_reset(rng=rng_, env_params=env_params)
  agent_state = reset_fn(train_state.params, init_timestep, rng_)
  runner_state = vbb.RunnerState(
    train_state=train_state,
    timestep=init_timestep,
    agent_state=agent_state,
    rng=rng,
  )
  _, transitions = vbb.collect_trajectory(
    runner_state=runner_state,
    num_steps=MAX_STEPS,
    actor_step_fn=actor.eval_step,
    env_step_fn=vmap_step,
    env_params=env_params,
  )
  transitions = jax.tree_util.tree_map(lambda x: jnp.swapaxes(x, 1, 0), transitions)
  return EpisodeData(
    timesteps=transitions.timestep,
    actions=transitions.action,
    transitions=transitions,
    positions=None,
    reaction_times=None,
  )


print(
  f"Algorithm set up (epsilon={EPSILON}). JIT will compile on first call, then reuse for all seeds."
)

Algorithm set up (epsilon=0.0). JIT will compile on first call, then reuse for all seeds.


In [10]:
# ---- Evaluate all seeds (JIT reused across seeds, multi-episode + multi-RNG) ----
def eval_all_configs(train_state, configs, config_type, rng_seeds):
  """Evaluate all configs, averaging over NUM_EPISODES episodes per RNG seed."""
  rows = []
  nparams = configs.world_seed.shape[0]
  for i in range(nparams):
    config_i = jax.tree_util.tree_map(lambda x: x[i : i + 1], configs)
    env_params = craftax_simulation_configs.make_multigoal_env_params(config_i)

    # Collect successes across all RNG seeds and all episodes
    all_successes = []
    all_path_lengths = []
    for rng_seed in rng_seeds:
      rng_i = jax.random.PRNGKey(rng_seed)
      episodes = eval_episode(rng_i, env_params, train_state)
      for j in range(NUM_EPISODES):
        episode = jax.tree_util.tree_map(lambda x: x[j], episodes)
        in_ep = get_in_episode(episode.timesteps)
        rewards = episode.timesteps.reward[in_ep]
        all_successes.append(float((rewards > 0.5).any()))
        all_path_lengths.append(int(in_ep.sum()))

    rows.append(
      {
        "config_type": config_type,
        "world_seed": int(configs.world_seed[i]),
        "success": np.mean(all_successes),
        "path_length": np.mean(all_path_lengths),
        "n_episodes": len(all_successes),
      }
    )
  return rows


rng_seeds = list(range(NUM_RNG_SEEDS))
all_results = []
for ckpt_seed in [1,2,3,5,6,7,10]:
  ckpt_path = os.path.join(DATA_DIR, MODEL_NAME, f"seed={ckpt_seed}")
  params = load_algorithm_ckpt_params(
    os.path.join(ckpt_path, f"{MODEL_NAME}.safetensors")
  )
  train_state = base_train_state.replace(params=params, target_network_params=params)

  rows = eval_all_configs(train_state, train_configs, "train_eval", rng_seeds)
  rows += eval_all_configs(train_state, test_configs, "test", rng_seeds)
  for r in rows:
    r["ckpt_seed"] = ckpt_seed
  all_results.extend(rows)

  avg_success = np.mean([r["success"] for r in rows])
  print(
    f"  seed={ckpt_seed}: avg_success={avg_success:.2f} ({NUM_EPISODES}ep x {NUM_RNG_SEEDS}rng per config)"
  )

results_df = pd.DataFrame(all_results)
print(f"\nDone. {len(results_df)} configs across {len(ckpt_seeds)} seeds.")
print(
  f"Each config evaluated over {NUM_EPISODES * NUM_RNG_SEEDS} total episodes (epsilon={EPSILON})."
)

  seed=1: avg_success=1.00 (10ep x 5rng per config)
  seed=2: avg_success=1.00 (10ep x 5rng per config)
  seed=3: avg_success=0.75 (10ep x 5rng per config)
  seed=5: avg_success=1.00 (10ep x 5rng per config)
  seed=6: avg_success=0.00 (10ep x 5rng per config)
  seed=7: avg_success=0.88 (10ep x 5rng per config)
  seed=10: avg_success=1.00 (10ep x 5rng per config)

Done. 56 configs across 1 seeds.
Each config evaluated over 50 total episodes (epsilon=0.0).


In [9]:
pivot = results_df.pivot_table(
  index=["ckpt_seed", "world_seed"],
  columns="config_type",
  values=["success", "path_length"],
  aggfunc="mean",
)

# Average success over ckpt_seed, per world_seed
avg_success = pivot["success"].groupby("ckpt_seed").mean()

# World seeds where any config_type has avg success < 1.0
ckpt_seeds = avg_success.index[avg_success.lt(1.0).any(axis=1)]
ckpt_seeds
# Filter
pivot.loc[pivot.index.get_level_values("ckpt_seed").isin(ckpt_seeds)]

path_length            success           
config_type                 test train_eval    test train_eval
ckpt_seed world_seed                                          
3         3                300.0     168.24     0.0        1.0
          15               300.0      46.52     0.0        1.0
          20                36.0      37.00     1.0        1.0
          95                34.0      29.00     1.0        1.0

---
## Epsilon Sweep

Re-run evaluation for a single checkpoint seed at different epsilon values to isolate the effect of exploration noise on success rate.

In [ ]:
# ---- Epsilon Sweep: evaluate one ckpt seed at multiple epsilon values ----
SWEEP_CKPT_SEED = 3  # pick a seed that showed failures
SWEEP_EPSILONS = [0.0, 0.1, 0.2]
SWEEP_NUM_EPISODES = 10
SWEEP_NUM_RNG_SEEDS = 5

# Load params for sweep seed
_sweep_ckpt_path = os.path.join(DATA_DIR, MODEL_NAME, f"seed={SWEEP_CKPT_SEED}")
_sweep_params = load_algorithm_ckpt_params(
  os.path.join(_sweep_ckpt_path, f"{MODEL_NAME}.safetensors")
)
_sweep_train_state = base_train_state.replace(
  params=_sweep_params, target_network_params=_sweep_params
)

sweep_results = []
for eps in SWEEP_EPSILONS:
  # Build actor with this epsilon
  _sweep_actor = make_epsilon_greedy_actor(
    config=arch_config, agent=agent, rng=setup_rng_, epsilon=eps
  )

  @jax.jit
  def _sweep_eval_episode(rng, env_params, train_state, _actor=_sweep_actor):
    rng, rng_ = jax.random.split(rng)
    init_timestep = vmap_reset(rng=rng_, env_params=env_params)
    agent_state = reset_fn(train_state.params, init_timestep, rng_)
    runner_state = vbb.RunnerState(
      train_state=train_state,
      timestep=init_timestep,
      agent_state=agent_state,
      rng=rng,
    )
    _, transitions = vbb.collect_trajectory(
      runner_state=runner_state,
      num_steps=MAX_STEPS,
      actor_step_fn=_actor.eval_step,
      env_step_fn=vmap_step,
      env_params=env_params,
    )
    transitions = jax.tree_util.tree_map(lambda x: jnp.swapaxes(x, 1, 0), transitions)
    return EpisodeData(
      timesteps=transitions.timestep,
      actions=transitions.action,
      transitions=transitions,
      positions=None,
      reaction_times=None,
    )

  for configs, config_type in [(train_configs, "train_eval"), (test_configs, "test")]:
    nparams = configs.world_seed.shape[0]
    for i in range(nparams):
      config_i = jax.tree_util.tree_map(lambda x: x[i : i + 1], configs)
      env_params = craftax_simulation_configs.make_multigoal_env_params(config_i)
      all_successes = []
      for rng_seed in range(SWEEP_NUM_RNG_SEEDS):
        rng_i = jax.random.PRNGKey(rng_seed)
        episodes = _sweep_eval_episode(rng_i, env_params, _sweep_train_state)
        for j in range(SWEEP_NUM_EPISODES):
          episode = jax.tree_util.tree_map(lambda x: x[j], episodes)
          in_ep = get_in_episode(episode.timesteps)
          rewards = episode.timesteps.reward[in_ep]
          all_successes.append(float((rewards > 0.5).any()))
      sweep_results.append(
        {
          "epsilon": eps,
          "config_type": config_type,
          "world_seed": int(configs.world_seed[i]),
          "success": np.mean(all_successes),
          "n_episodes": len(all_successes),
        }
      )
  print(
    f"  epsilon={eps}: avg_success={np.mean([r['success'] for r in sweep_results if r['epsilon'] == eps]):.3f}"
  )

sweep_df = pd.DataFrame(sweep_results)

# Plot
fig, ax = plt.subplots(1, 1, figsize=(8, 5))
sweep_summary = (
  sweep_df.groupby("epsilon")["success"].agg(["mean", "std"]).reset_index()
)
ax.bar(
  [str(e) for e in sweep_summary["epsilon"]],
  sweep_summary["mean"],
  yerr=sweep_summary["std"],
  capsize=5,
  color=["#2ecc71", "#f39c12", "#e74c3c"],
  edgecolor="black",
)
ax.set_xlabel("Epsilon")
ax.set_ylabel("Mean Success Rate")
ax.set_title(
  f"Epsilon Sweep (ckpt seed={SWEEP_CKPT_SEED}, {SWEEP_NUM_EPISODES}ep x {SWEEP_NUM_RNG_SEEDS}rng)"
)
ax.set_ylim(0, 1.05)
ax.axhline(1.0, color="gray", linestyle="--", alpha=0.5, label="Perfect")
ax.legend()
plt.tight_layout()
plt.show()

# Detail table
print("\nPer-config breakdown:")
print(
  sweep_df.pivot_table(
    index=["config_type", "world_seed"], columns="epsilon", values="success"
  )
)

---
## Single Seed Debug (visualize paths)

Set `DEBUG_SEED` below and run the remaining cells to inspect a specific checkpoint.

In [22]:
DEBUG_SEED = 3  # which ckpt seed to visualize

In [25]:
# ---- Inspect param dtypes for DEBUG_SEED ----
ckpt_path = os.path.join(DATA_DIR, MODEL_NAME, f"seed={DEBUG_SEED}")
params = load_algorithm_ckpt_params(
  os.path.join(ckpt_path, f"{MODEL_NAME}.safetensors")
)

print(f"--- Parameter Inspection (seed={DEBUG_SEED}) ---")
for path, leaf in jax.tree_util.tree_leaves_with_path(params):
  path_str = "/".join(str(p) for p in path)
  leaf = np.asarray(leaf)
  has_nan = np.any(np.isnan(leaf))
  has_inf = np.any(np.isinf(leaf))
  if has_nan or has_inf:
    print(
      f"  {path_str:60s} | dtype={leaf.dtype} | shape={str(leaf.shape):20s} "
      f"| mean={leaf.mean():.4f} | std={leaf.std():.4f} "
      f"| nan={has_nan} | inf={has_inf}"
    )

--- Parameter Inspection (seed=3) ---


In [26]:
# ---- Load params & run eval for visualization ----
ckpt_path = os.path.join(DATA_DIR, MODEL_NAME, f"seed={DEBUG_SEED}")
params = load_algorithm_ckpt_params(
  os.path.join(ckpt_path, f"{MODEL_NAME}.safetensors")
)
train_state = base_train_state.replace(params=params, target_network_params=params)

viz_episodes = []
for configs, config_type in [(train_configs, "train_eval"), (test_configs, "test")]:
  nparams = configs.world_seed.shape[0]
  for i in range(nparams):
    config_i = jax.tree_util.tree_map(lambda x: x[i : i + 1], configs)
    env_params = craftax_simulation_configs.make_multigoal_env_params(config_i)
    episodes = eval_episode(rng, env_params, train_state)
    episodes = episodes._replace(positions=get_agent_position(episodes.timesteps))
    episode = jax.tree_util.tree_map(lambda x: x[0], episodes)
    in_ep = get_in_episode(episode.timesteps)
    episode = jax.tree_util.tree_map(lambda x: x[in_ep], episode)
    viz_episodes.append((episode, int(configs.world_seed[i]), config_type, i))
print(f"Collected {len(viz_episodes)} episodes for visualization")

Collected 8 episodes for visualization


In [ ]:
# ---- Visualize Paths Per World Seed ----
unique_seeds = sorted(set(ws for _, ws, _, _ in viz_episodes))

for seed in unique_seeds:
  seed_episodes = [(ep, ct, ci) for ep, ws, ct, ci in viz_episodes if ws == seed]
  n_panels = len(seed_episodes)
  fig, axs = plt.subplots(1, n_panels, figsize=(5 * n_panels, 5))
  if n_panels == 1:
    axs = [axs]

  for ax, (episode, config_type, config_idx) in zip(axs, seed_episodes):
    first_state = jax.tree_util.tree_map(lambda x: x[0], episode.timesteps.state)
    with jax.disable_jit():
      image = render_fn(first_state, show_agent=False)
      path = episode.positions
      actions = actions_from_path(path)
      place_arrows_on_image(
        image=image,
        positions=path,
        actions=actions,
        maze_height=first_state.map.shape[1],
        maze_width=first_state.map.shape[2],
        ax=ax,
        display_image=True,
        arrow_color="red",
        show_path_length=False,
        start_color="red",
      )

    rewards = episode.timesteps.reward
    ep_success = float((rewards > 0.5).any())
    ep_length = len(episode.positions)
    status = "SUCCESS" if ep_success else "FAIL"
    ax.set_title(
      f"{config_type} (idx={config_idx})\n{status} | len={ep_length}",
      fontsize=11,
    )

  fig.suptitle(
    f"World Seed {seed} (ckpt seed={DEBUG_SEED})",
    fontsize=14,
    fontweight="bold",
  )
  plt.tight_layout()
  plt.show()

In [ ]:
# ---- Env Params & Observation Dtype Inspection ----
print("--- Env Params Dtypes ---")
for path, leaf in jax.tree_util.tree_leaves_with_path(example_env_params):
  path_str = "/".join(str(p) for p in path)
  if hasattr(leaf, "dtype"):
    print(f"  {path_str:60s} | dtype={leaf.dtype} | shape={leaf.shape}")

print("\n--- Example Timestep Observation Dtypes ---")
for path, leaf in jax.tree_util.tree_leaves_with_path(example_timestep):
  path_str = "/".join(str(p) for p in path)
  if hasattr(leaf, "dtype"):
    print(f"  {path_str:60s} | dtype={leaf.dtype} | shape={leaf.shape}")